In [1]:
from dotenv import load_dotenv, find_dotenv 
load_dotenv(find_dotenv())

True

In [2]:
from openai import OpenAI

client = OpenAI()

In [3]:
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Cual es la capital de Francia?"},
    ],
)

In [4]:
print(response.choices[0].message.content)

La capital de Francia es París.


In [5]:
from litellm import completion

In [6]:
response = completion(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Cual es la capital de Francia?"},
    ],
)

In [7]:
print(response.choices[0].message.content)

La capital de Francia es París.


In [ ]:
# mantener una conversación con el modelo (mantener el contexto en una estructura de datos de mensajes)

def call_to_llm(messages: list[dict[str, str]]) -> str:
    r = completion(
        model=MODEL,
        messages=messages,
    )
    llm_response = r.choices[0].message.content
    messages.append({"role": "assistant", "content": llm_response})
    return llm_response

MODEL = "gpt-5-mini"
messages = []

messages.append({"role": "system", "content": "Eres un asistente útil."})
messages.append({"role": "user", "content": "Mi nombre es Juan, ¿puedes saludarme?"})

response1 = call_to_llm(messages)
print(response1)

messages.append({"role": "user", "content": "¿Cuál es mi nombre?"})
response2 = call_to_llm(messages)
print(response2)

¡Hola, Juan! ¿Cómo estás? ¿En qué puedo ayudarte hoy?
Tu nombre es Juan. ¿Quieres que lo recuerde para algo más?


In [11]:
messages

[{'role': 'system', 'content': 'Eres un asistente útil.'},
 {'role': 'user', 'content': 'Mi nombre es Juan, ¿puedes saludarme?'},
 {'role': 'assistant',
  'content': '¡Hola, Juan! ¿Cómo estás? ¿En qué puedo ayudarte hoy?'},
 {'role': 'user', 'content': '¿Cuál es mi nombre?'},
 {'role': 'assistant',
  'content': 'Tu nombre es Juan. ¿Quieres que lo recuerde para algo más?'}]

In [ ]:
## output formateado como JSON
from pydantic import BaseModel
from litellm import completion

class ExtratecInfo(BaseModel):
    name: str
    email: str
    phone: str

response = completion(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Extrae la información de contacto de este texto: 'Mi nombre es Juan Pérez, mi correo es juan.perez@example.com y mi teléfono es 123-456-7890'."}
    ],
    response_format=ExtratecInfo ## Que potente es esto, podemos definir un modelo de salida y el LLM nos devolverá un objeto de ese tipo, en este caso un objeto de tipo ExtratecInfo
)

print(response.choices[0].message.content)


{"name":"Juan Pérez","email":"juan.perez@example.com","phone":"123-456-7890"}


## LLamadas asincronas

In [ ]:
import asyncio
from litellm import acompletion # usar acompletion para llamadas asincrónicas al LLM

N = 10  # Número de solicitudes concurrentes
semaphore = asyncio.Semaphore(N)  # Limitar a N solicitudes concurrentes

async def call_llm(prompt: str) -> str:
    async with semaphore:
        response = await acompletion(
            model="gpt-5-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt},
            ],
            num_retries=3  # Limitar a 1 respuesta por solicitud
        )
        return response.choices[0].message.content
    
prompts = [f"Cuanto es {i} + {i}" for i in range(20)]  # 20 prompts de ejemplo
tasks = [call_llm(prompt) for prompt in prompts]
results = await asyncio.gather(*tasks, return_exceptions=True)

In [22]:
print(results)

['0 + 0 = 0.\n\nLa suma de cero y cero es cero porque no hay cantidad añadida.', '1 + 1 = 2.', '2 + 2 = 4.', '3 + 3 = 6. ¿Quieres que te explique cómo se suma?', '4 + 4 = 8.', '5 + 5 = 10.', '6 + 6 = 12.', '7 + 7 = 14.', '8 + 8 = 16.', '9 + 9 = 18.', '10 + 10 = 20.', '11 + 11 = 22.', '12 + 12 = 24.', '13 + 13 = 26.', '14 + 14 = 28.', '15 + 15 = 30.', '16 + 16 = 32.', '17 + 17 = 34.', '18 + 18 = 36.', '19 + 19 = 38.']
